In [19]:
import numpy as np

from pathlib import Path
import torch
from torch.utils.data import DataLoader
import torch.nn as nn



current_dir = Path.cwd()

if "ESPECT_CONV" in current_dir.parts:
    idx = current_dir.parts.index("ESPECT_CONV")
    PROJECT_ROOT = Path(*current_dir.parts[:idx + 1])
else:
    PROJECT_ROOT = current_dir

INPUT_DIR = PROJECT_ROOT / "processed_data" / "stage_30_array_assembly"
OUTPUT_DIR = PROJECT_ROOT / "processed_data" / "stage_40_cnn"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print("Raiz do Projeto:", PROJECT_ROOT)
print("Entrada:", INPUT_DIR)
print("Saída:", OUTPUT_DIR)

Raiz do Projeto: /home/jobson/Documentos/ESPECT_CONV
Entrada: /home/jobson/Documentos/ESPECT_CONV/processed_data/stage_30_array_assembly
Saída: /home/jobson/Documentos/ESPECT_CONV/processed_data/stage_40_cnn


In [7]:

power_array = np.load(
    INPUT_DIR / "sub-01_ses-01_power.npy"
)

labels = np.load(
    INPUT_DIR / "sub-01_ses-01_labels.npy"
)

print("Power:", power_array.shape)
print("Labels:", labels.shape)

Power: (80, 21, 21, 65, 9)
Labels: (80,)


In [8]:
power_array = np.transpose(
    power_array,
    (0, 4, 3, 1, 2)
)

print(power_array.shape)

(80, 9, 65, 21, 21)


In [11]:
X = torch.from_numpy(
    power_array
).float()

y = torch.from_numpy(
    labels
).long()

print(X.shape)
print(y.shape)

torch.Size([80, 9, 65, 21, 21])
torch.Size([80])


## Criando o dataset

In [12]:
from torch.utils.data import TensorDataset

dataset = TensorDataset(
    X,
    y
)

x, label = dataset[0]

print(x.shape)
print(label)

torch.Size([9, 65, 21, 21])
tensor(1)


## Criando o DataLoader

In [15]:

loader = DataLoader(
    dataset,
    batch_size=8,
    shuffle=True
)

for X_batch, y_batch in loader:

    print(X_batch.shape)
    print(y_batch.shape)

    

torch.Size([8, 9, 65, 21, 21])
torch.Size([8])
torch.Size([8, 9, 65, 21, 21])
torch.Size([8])
torch.Size([8, 9, 65, 21, 21])
torch.Size([8])
torch.Size([8, 9, 65, 21, 21])
torch.Size([8])
torch.Size([8, 9, 65, 21, 21])
torch.Size([8])
torch.Size([8, 9, 65, 21, 21])
torch.Size([8])
torch.Size([8, 9, 65, 21, 21])
torch.Size([8])
torch.Size([8, 9, 65, 21, 21])
torch.Size([8])
torch.Size([8, 9, 65, 21, 21])
torch.Size([8])
torch.Size([8, 9, 65, 21, 21])
torch.Size([8])


## Criando a Rede CNN

In [ ]:
class SpaceFrequencyCNN(nn.Module):

    def __init__(self):

        super().__init__()

        self.conv1 = nn.Conv3d(
            in_channels=9,
            out_channels=16,
            kernel_size=3,
            padding=1
        )

        self.relu1 = nn.ReLU()

        self.pool1 = nn.MaxPool3d(
            kernel_size=2
        )

        self.conv2 = nn.Conv3d(
            in_channels=16,
            out_channels=8,
            kernel_size=3,
            padding=1
        )

        self.relu2 = nn.ReLU()

        self.pool2 = nn.MaxPool3d(
            kernel_size=2
        )

        self.flatten = nn.Flatten()

        self.latent = nn.Linear(
            8 * 16 * 5 * 5,
            3
        )


    def forward(self, x):

        x = self.conv1(x)
        x = self.relu1(x)
        x = self.pool1(x)

        x = self.conv2(x)
        x = self.relu2(x)
        x = self.pool2(x)

        x = self.flatten(x)

        x = self.latent(x)

        return x